In [ ]:
import pandas as pd
import numpy as np
import mne
from pathlib import Path
import re

# -----------------------------
# CSV -> FIF converter
# -----------------------------
csv_path = Path("adhdata.csv")          
out_dir = Path("New_data/out_fif")
out_dir.mkdir(parents=True, exist_ok=True)

sfreq = 128  

df = pd.read_csv(csv_path)

if df.shape[1] == 1:
    df = df.iloc[:, 0].str.split(",", expand=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

channel_cols = [c for c in df.columns if c not in ["Class", "ID"]]

for subject_id, subject_df in df.groupby("ID"):
    group = subject_df["Class"].iloc[0]

    # v10p -> 10
    vid = int(re.search(r"\d+", str(subject_id)).group())

    # MNE expects Volts. CSV values look like microvolts, so convert µV -> V
    data = subject_df[channel_cols].astype(float).to_numpy().T * 1e-6

    info = mne.create_info(
        ch_names=channel_cols,
        sfreq=sfreq,
        ch_types="eeg"
    )

    raw = mne.io.RawArray(data, info, verbose=False)

    try:
        raw.set_montage("standard_1020", on_missing="ignore")
    except Exception:
        pass

    fif_name = f"v{vid}p_class-{group}_raw.fif"
    raw.save(out_dir / fif_name, overwrite=True, verbose=False)



Saved 121 FIF files to New_data\out_fif
